In [1]:
# import needed libraries
import os
import numpy as np
import open3d as o3d

# import custom libraries
import al_utils as al

In [2]:
# Configuration
USED_PCD_ID = 300
USED_PCD_FILE_DIR = 'used_pcd'
SRC_PCD_FILE_DIR = 'points'

In [7]:
# get current working directory and find pcd files
print('Working directory:', os.getcwd())
path_points = os.path.join(os.getcwd(), SRC_PCD_FILE_DIR)

pcd_file = al.get_target_pcd(path_points, USED_PCD_ID)
print(f'Using No.{USED_PCD_ID} pcd file: {pcd_file}')

Working directory: /root/AC1/autopath
Path is valid
Total 741 pcd files found, total size: 475.11 MB
Target pcd file: 1771498195.700643062.pcd, size: 658.27 KB
Using No.300 pcd file: /root/AC1/autopath/points/1771498195.700643062.pcd


In [ ]:
# get pcd file name from full path
pcd_file_name = os.path.basename(pcd_file)
print(f'Using No.{USED_PCD_ID} pcd file: {pcd_file_name}')

# copy pcd file to used_pcd folder if it doesn't exist or is different file
def copy_used_pcd(used_pcd_path, pcd_file_path):
    print('Copy pcd file to used_pcd folder...')
    if not os.path.exists(os.path.join(os.getcwd(), used_pcd_path)):
        print('Create used_pcd folder')
        os.makedirs(os.path.join(os.getcwd(), used_pcd_path))

    # Check if the file already exists in the used_pcd folder
    if os.path.exists(os.path.join(os.getcwd(), used_pcd_path, os.path.basename(pcd_file_path))):
        print(f'File {os.path.basename(pcd_file_path)} already exists in used_pcd folder, Checking if it is the same file...')
        
        # check md5sum

Using No.300 pcd file: 1771498195.700643062.pcd


In [10]:
def copy_used_pcd():
    print('Copy pcd file to used_pcd folder...')
    if not os.path.exists(os.getcwd() + '/used_pcd'):
        print('Create used_pcd folder')
        os.makedirs(os.getcwd() + '/used_pcd')

    # Check if the file already exists in the used_pcd folder
    if os.path.exists(os.getcwd() + '/used_pcd/' + pcd_file_name):
        print(f'File {pcd_file_name} already exists in used_pcd folder, Checking if it is the same file...')
        if os.path.getsize(os.getcwd() + '/used_pcd/' + pcd_file_name) == os.path.getsize(path_points + '/' + pcd_file_name):
            print(f'File {pcd_file_name} is the same file, no need to copy')
        else:
            print(f'File {pcd_file_name} is different file, copying...')
            os.remove(os.getcwd() + '/used_pcd/' + pcd_file_name)
            os.rename(path_points + '/' + pcd_file_name, os.getcwd() + '/used_pcd/' + pcd_file_name)
    else:
        print(f'File {pcd_file_name} does not exist in used_pcd folder, copying...')
        # open source file in binary mode
        with open(path_points + '/' + pcd_file_name, 'rb') as f:
            # open destination file in binary mode
            with open(os.getcwd() + '/used_pcd/' + pcd_file_name, 'wb') as f2:
                # copy the contents of the source file to the destination file
                f2.write(f.read())

In [11]:
copy_used_pcd()

Copy pcd file to used_pcd folder...
File 1771498195.700643062.pcd already exists in used_pcd folder, Checking if it is the same file...
File 1771498195.700643062.pcd is the same file, no need to copy


In [ ]:
# # cpoy target pcd file to used_pcd folder
# if not os.path.exists(os.getcwd() + '/used_pcd'):
#     print('Create used_pcd folder')
#     os.makedirs(os.getcwd() + '/used_pcd')

# # Check if the file already exists in the used_pcd folder
# if os.path.exists(os.getcwd() + '/used_pcd/' + pcd_file_name):
#     print(f'File {pcd_file_name} already exists in used_pcd folder, Checking if it is the same file...')
#     if os.path.getsize(os.getcwd() + '/used_pcd/' + pcd_file_name) == os.path.getsize(path_points + '/' + pcd_file_name):
#         print(f'File {pcd_file_name} is the same file, no need to copy')
#     else:
#         print(f'File {pcd_file_name} is different file, copying...')
#         os.remove(os.getcwd() + '/used_pcd/' + pcd_file_name)
#         os.rename(path_points + '/' + pcd_file_name, os.getcwd() + '/used_pcd/' + pcd_file_name)
# else:
#     print(f'File {pcd_file_name} does not exist in used_pcd folder, copying...')
#     # open source file in binary mode
#     with open(path_points + '/' + pcd_file_name, 'rb') as source:
#         # open destination file in binary mode
#         with open(os.getcwd() + '/used_pcd/' + pcd_file_name, 'wb') as destination:
#             # copy the contents of the source file to the destination file
#             destination.write(source.read())


File 1771498195.700643062.pcd already exists in used_pcd folder, Checking if it is the same file...
File 1771498195.700643062.pcd is the same file, no need to copy


In [5]:
# read pcd file
pcd = o3d.io.read_point_cloud(os.getcwd() + '/used_pcd/' + pcd_file_name)

# print pcd info
total_points = len(pcd.points)

# remove nan points and infinite points
pcd = pcd.remove_non_finite_points(remove_nan=True, remove_infinite=True)

# print pcd info
print(f"Valid points: {len(pcd.points)}, valid ratio: {len(pcd.points) / total_points * 100:.2f}%")
print(f"Normals: {len(pcd.normals)}")

Valid points: 17628, valid ratio: 63.76%
Normals: 0


In [6]:
# utils

# Add color to point cloud
def add_color(pcd, color=[1, 0, 0]):
    num_points = len(pcd.points)
    color_array = np.ones((num_points, 3)) * color
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    return pcd

# Show point cloud
def show_point_cloud(pcd, name = "Point Cloud"):
    o3d.visualization.draw_geometries([pcd], window_name=name)

# Change the color of point cloud
def change_color(pcd, ids=[], color=[0.5, 0.5, 0.5]):
    colors = np.array(pcd.colors)
    for i in ids:
        colors[i] = color
    pcd.colors = o3d.utility.Vector3dVector(colors)
    return pcd

In [7]:
# merge point clouds
def merge_pcds(pcd1, pcd2):
    """
    Open3D官方风格的点云拼接函数
    :param pcd1: 第一个点云（o3d.geometry.PointCloud）
    :param pcd2: 第二个点云（o3d.geometry.PointCloud）
    :return: 拼接后的新点云
    """
    # 1. 创建空点云用于存储拼接结果
    merged_pcd = o3d.geometry.PointCloud()

    # ---------------- 核心：拼接坐标点（必选） ----------------
    # 将Open3D的Vector3dVector转为NumPy数组（零拷贝，高效）
    points1 = np.asarray(pcd1.points)
    points2 = np.asarray(pcd2.points)
    # 按行拼接（行数=点数，列数=3（x,y,z））
    merged_points = np.vstack([points1, points2])
    # 赋值回新点云
    merged_pcd.points = o3d.utility.Vector3dVector(merged_points)

    # ---------------- 可选：拼接颜色（有则拼，无则跳过） ----------------
    if pcd1.has_colors() and pcd2.has_colors():
        colors1 = np.asarray(pcd1.colors)
        colors2 = np.asarray(pcd2.colors)
        merged_colors = np.vstack([colors1, colors2])
        merged_pcd.colors = o3d.utility.Vector3dVector(merged_colors)
    elif pcd1.has_colors():
        # 仅pcd1有颜色，给pcd2补默认灰色（0.5,0.5,0.5）
        colors1 = np.asarray(pcd1.colors)
        colors2 = np.ones((len(points2), 3)) * 0.5
        merged_colors = np.vstack([colors1, colors2])
        merged_pcd.colors = o3d.utility.Vector3dVector(merged_colors)
    elif pcd2.has_colors():
        # 仅pcd2有颜色，给pcd1补默认灰色
        colors1 = np.ones((len(points1), 3)) * 0.5
        colors2 = np.asarray(pcd2.colors)
        merged_colors = np.vstack([colors1, colors2])
        merged_pcd.colors = o3d.utility.Vector3dVector(merged_colors)

    # ---------------- 可选：拼接法向量（需两个点云都有） ----------------
    if pcd1.has_normals() and pcd2.has_normals():
        normals1 = np.asarray(pcd1.normals)
        normals2 = np.asarray(pcd2.normals)
        merged_normals = np.vstack([normals1, normals2])
        merged_pcd.normals = o3d.utility.Vector3dVector(merged_normals)

    return merged_pcd

In [30]:
# add color to point cloud
# add_color(pcd, color=[0.5, 0.5, 0.5])
# show_point_cloud(pcd, name="Original Point Cloud")

# # change the color of point cloud
# change_color(pcd, [0, 1, 2, 3, 4, 5, 6, 7, 5000, 5001, 5002], [0, 1, 0])
# show_point_cloud(pcd, name="Modified Point Cloud")


In [8]:
# Downsample
downpcd = pcd.voxel_down_sample(voxel_size=0.1)
# o3d.visualization.draw_geometries([downpcd])
# print downpcd info
print(f"Downsampled points: {len(downpcd.points)}, downsample ratio: {len(downpcd.points) / len(pcd.points) * 100:.2f}%", )

# show_point_cloud(downpcd, name=f"Downsampled Point Cloud: {len(downpcd.points)} points")

# Remove outliers
cl, ind = downpcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=1.0)
filtered_pcd = downpcd.select_by_index(ind)

# Visualize if needed
show_point_cloud(filtered_pcd, name=f"Filtered Point Cloud: {len(filtered_pcd.points)} points")

print(f"Filtered points: {len(filtered_pcd.points)}, Filtered ratio: {len(filtered_pcd.points) / len(downpcd.points) * 100:.2f}%")


Downsampled points: 8180, downsample ratio: 46.40%
Filtered points: 7185, Filtered ratio: 87.84%


In [9]:
planes = al.find_planes(filtered_pcd, distance_threshold=0.05, ransac_n=3, num_iterations=1000)

print(f"Found {len(planes)} planes")

Found 49 planes


In [65]:
# show all planes
color_list = [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 0, 1], [0, 1, 1], [1, 0.5, 0], [0.5, 1, 0], [0.5, 0, 1],[0.5, 0.5, 1],[1, 0.5, 0.5], [0.5, 1, 0.5], [0.5, 0.5, 1]]
planes_pcd = o3d.geometry.PointCloud()
minimum_points = 50
plane_cnt = 0
for i in range(len(planes)):
    plane_model, plane_pcd = planes[i]

    # only keep the plane horizontal
    [a, b, c, d] = plane_model
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)
    z_component = abs(normal[2])
    if z_component < 0.9:
        continue

    # skip small planes
    if len(plane_pcd.points) < minimum_points:
        continue

    # show_point_cloud(plane_pcd, name=f"Plane Point Cloud: {len(plane_pcd.points)} points")
    add_color(plane_pcd, color=color_list[i % len(color_list)])
    planes_pcd = merge_pcds(planes_pcd, plane_pcd)
    plane_cnt += 1

show_point_cloud(planes_pcd, name=f"Total {plane_cnt} planes: {len(planes_pcd.points)} points")
    

In [ ]:
# Use RANSAC to fit a plane
plane_model, inliers = filtered_pcd.segment_plane(
    distance_threshold=0.05,
    ransac_n=3,
    num_iterations=1000
)
[a, b, c, d] = plane_model
print(f"Equation of the best-fit plane: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")

# 筛选水平面（法向量z分量绝对值接近1，即法向量接近Z轴）
normal = np.array([a, b, c])
normal = normal / np.linalg.norm(normal)  # 归一化法向量
z_component = abs(normal[2])
if z_component < 0.9:  # 法向量z分量小于0.9，认为不是水平面
    raise ValueError(f"检测到的平面不是水平面（法向量z分量: {z_component:.4f}）")

# G
plane_pcd = filtered_pcd.select_by_index(inliers)

change_color(filtered_pcd, inliers, color=[0, 0.75, 0])
show_point_cloud(filtered_pcd)

# 计算平面中心坐标
plane_points = np.asarray(plane_pcd.points)
plane_center = np.mean(plane_points, axis=0)
print(f"水平面中心坐标: {plane_center}")

拟合平面方程: -0.1466x + 0.1016y + 0.9840z + 1.8489 = 0
水平面中心坐标: [ 4.06304405 -0.98687851 -1.17189911]


In [ ]:
import open3d as o3d
import numpy as np

def find_all_planes(
    pcd,
    distance_threshold=0.05,
    ransac_n=3,
    num_iterations=1000,
    min_plane_points=100,  # 平面最小点数（低于此数视为无效平面）
    horizontal_only=True,  # 是否仅筛选水平面
    z_component_thresh=0.9  # 水平面法向量Z分量阈值
):
    """
    迭代分割点云中所有的平面（可选仅保留水平面）
    :param pcd: 输入点云（Open3D PointCloud对象）
    :param min_plane_points: 有效平面的最小点数（避免检测到噪点组成的小平面）
    :param horizontal_only: 是否只保留水平面
    :return: 所有有效平面的列表，每个元素包含：(plane_model, inliers, plane_center)
    """
    # 复制点云，避免修改原始数据
    remaining_pcd = pcd.select_by_index(list(range(len(pcd.points))))
    all_planes = []  # 存储所有找到的平面
    
    while True:
        # 1. 对剩余点云执行RANSAC平面分割
        try:
            plane_model, inliers = remaining_pcd.segment_plane(
                distance_threshold=distance_threshold,
                ransac_n=ransac_n,
                num_iterations=num_iterations
            )
        except:
            # 无法分割出平面，终止迭代
            break
        
        # 2. 过滤无效平面（点数太少）
        if len(inliers) < min_plane_points:
            break
        
        # 3. 若仅筛选水平面，验证法向量Z分量
        if horizontal_only:
            [a, b, c, d] = plane_model
            normal = np.array([a, b, c])
            normal = normal / np.linalg.norm(normal)  # 归一化法向量
            z_component = abs(normal[2])
            if z_component < z_component_thresh:
                # 不是水平面，移除这些点后继续迭代（不保存该平面）
                remaining_pcd = remaining_pcd.select_by_index(inliers, invert=True)
                continue
        
        # 4. 计算该平面的中心坐标
        plane_pcd = remaining_pcd.select_by_index(inliers)
        plane_points = np.asarray(plane_pcd.points)
        plane_center = np.mean(plane_points, axis=0)
        
        # 5. 保存该平面的信息
        all_planes.append({
            "plane_model": plane_model,  # 平面方程参数 [a,b,c,d]
            "inliers": inliers,          # 该平面的点索引（相对剩余点云）
            "global_inliers": None,      # 后续补充：相对原始点云的索引
            "plane_center": plane_center # 平面中心坐标
        })
        
        # 6. 从剩余点云中移除该平面的点，继续找下一个平面
        remaining_pcd = remaining_pcd.select_by_index(inliers, invert=True)
        
        # 7. 终止条件：剩余点云点数不足最小平面点数
        if len(remaining_pcd.points) < min_plane_points:
            break
    
    # 补充：将“相对剩余点云的索引”转换为“相对原始点云的索引”
    # （因为每次移除点后，剩余点云的索引会变化，需要回溯）
    # global_indices = list(range(len(pcd.points)))  # 原始点云索引
    # for plane in all_planes:
    #     # 该平面的索引是当前剩余点云的索引，映射到原始索引
    #     plane["global_inliers"] = [global_indices[idx] for idx in plane["inliers"]]
    #     # 更新剩余原始索引（移除已找到的平面点）
    #     global_indices = [global_indices[idx] for idx in range(len(remaining_pcd.points))]
    
    return all_planes


if __name__ == "__main__":
    # 1. 加载/生成测试点云（替换为你的点云路径）
    # pcd = o3d.io.read_point_cloud("your_point_cloud.pcd")
    # 模拟多平面点云（3个水平面 + 随机噪点）
    def create_multi_plane_pcd():
        # 平面1：z=0，1000个点
        plane1 = np.hstack([np.random.rand(1000, 2)*10 - 5, np.zeros((1000, 1))])
        # 平面2：z=1，800个点
        plane2 = np.hstack([np.random.rand(800, 2)*10 - 5, np.ones((800, 1))])
        # 平面3：z=2，500个点
        plane3 = np.hstack([np.random.rand(500, 2)*10 - 5, 2*np.ones((500, 1))])
        # 噪点：随机分布
        noise = np.random.rand(200, 3)*10 - 5
        # 合并所有点
        all_points = np.vstack([plane1, plane2, plane3, noise])
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(all_points)
        return pcd
    
    pcd = create_multi_plane_pcd()
    
    # 2. 找到所有水平面
    all_planes = find_all_planes(
        pcd,
        distance_threshold=0.05,
        min_plane_points=200,  # 至少200个点才视为有效平面
        horizontal_only=True,
        z_component_thresh=0.9
    )
    
    # 3. 打印所有平面信息
    print(f"共找到 {len(all_planes)} 个有效平面：")
    for i, plane in enumerate(all_planes):
        [a, b, c, d] = plane["plane_model"]
        print(f"\n平面 {i+1}：")
        print(f"  平面方程：{a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
        # print(f"  平面点数：{len(plane['global_inliers'])}")
        print(f"  平面中心：{plane['plane_center']}")
    

共找到 3 个有效平面：

平面 1：
  平面方程：-0.0000x + -0.0000y + 1.0000z + -0.0000 = 0
  平面中心：[-0.16791086 -0.03949701  0.        ]

平面 2：
  平面方程：-0.0000x + -0.0000y + 1.0000z + -1.0001 = 0
  平面中心：[0.05434611 0.07439917 1.00006012]

平面 3：
  平面方程：-0.0000x + 0.0000y + 1.0000z + -2.0000 = 0
  平面中心：[0.07336541 0.06251202 1.99997051]


In [ ]:
import matplotlib.pyplot as plt

# -------------------------- 2. 检测水平面 --------------------------
def detect_horizontal_plane(pcd, distance_threshold=0.05, ransac_n=3, num_iterations=1000):
    """
    从点云中检测水平面（法向量接近Z轴）
    :param pcd: 预处理后的点云对象
    :param distance_threshold: 点到平面的距离阈值
    :param ransac_n: RANSAC每次采样的点数
    :param num_iterations: RANSAC迭代次数
    :return: 平面参数（ax+by+cz+d=0）、平面点云、平面中心坐标
    """
    # 使用RANSAC拟合平面
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )
    [a, b, c, d] = plane_model
    print(f"拟合平面方程: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
    
    # 筛选水平面（法向量z分量绝对值接近1，即法向量接近Z轴）
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)  # 归一化法向量
    z_component = abs(normal[2])
    if z_component < 0.9:  # 法向量z分量小于0.9，认为不是水平面
        raise ValueError(f"检测到的平面不是水平面（法向量z分量: {z_component:.4f}）")
    
    # 提取平面内的点云
    plane_pcd = pcd.select_by_index(inliers)
    # plane_pcd.paint_uniform_color([1, 0, 0])  # 红色标记水平面
    
    # 计算平面中心坐标
    plane_points = np.asarray(plane_pcd.points)
    plane_center = np.mean(plane_points, axis=0)
    print(f"水平面中心坐标: {plane_center}")
    
    return plane_model, plane_pcd, plane_center

# -------------------------- 3. 规划降落路径 --------------------------
def plan_landing_path(plane_center, safe_height=2.0, step_num=50):
    """
    规划无人机降落路径：原点 → 目标点正上方（安全高度） → 垂直降落至平面中心
    :param plane_center: 水平面中心坐标 [x, y, z]
    :param safe_height: 目标点正上方的安全高度（相对于平面）
    :param step_num: 路径点数量（均分）
    :return: 降落路径点云、路径点坐标数组
    """
    # 路径关键点
    start_point = np.array([0, 0, 0])  # 起点（原点）
    hover_point = np.array([plane_center[0], plane_center[1], plane_center[2] + safe_height])  # 悬停点（安全高度）
    land_point = plane_center  # 降落点（平面中心）
    
    # 生成分段平滑路径
    # 第一段：原点到悬停点（直线）
    path_1 = np.linspace(start_point, hover_point, step_num // 2)
    # 第二段：悬停点到降落点（垂直下降）
    path_2 = np.linspace(hover_point, land_point, step_num // 2)
    # 合并路径
    path_points = np.vstack((path_1, path_2))
    
    # 创建路径点云（蓝色标记）
    path_pcd = o3d.geometry.PointCloud()
    path_pcd.points = o3d.utility.Vector3dVector(path_points)
    path_pcd.paint_uniform_color([0, 0, 1])  # 蓝色标记路径
    
    print(f"降落路径规划完成：")
    print(f"  - 起点: {start_point}")
    print(f"  - 悬停点: {hover_point} (安全高度: {safe_height}m)")
    print(f"  - 降落点: {land_point}")
    
    return path_pcd, path_points

# -------------------------- 4. 可视化结果 --------------------------
def visualize_result(pcd, plane_pcd, path_pcd):
    """
    可视化原始点云、水平面、降落路径
    """
    # 原始点云设为灰色
    pcd.paint_uniform_color([0.5, 0.5, 0.5])
    # 创建坐标系（原点）
    coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
    # 可视化
    o3d.visualization.draw_geometries([pcd, plane_pcd, path_pcd, coordinate_frame],
                                      window_name="点云水平面检测与降落路径规划",
                                      width=1280, height=720)


In [8]:
# 检测水平面
plane_model, plane_pcd, plane_center = detect_horizontal_plane(filtered_pcd)

# 规划降落路径
path_pcd, path_points = plan_landing_path(plane_center, safe_height=2.0)

# 可视化结果
visualize_result(pcd, plane_pcd, path_pcd)

拟合平面方程: -0.1446x + 0.1025y + 0.9842z + 1.8427 = 0


: 